# CVRIE - Supervised Learning: Brain Tumor Classification

## 1. Introduction & Context

### **Project context**

This notebook covers the supervised part of the CVRIE project. The hospital wants CVRIE
(a medical AI) to be able to analyze medical images and help with diagnosis. Our goal is to
train a classification model able to tell apart different types of brain scans from MRI images.

### **Dataset choice**

We picked the [Brain Tumor MRI Dataset](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset) on Kaggle, which contains brain MRI scans split into 4 classes:
- **Glioma**
- **Meningioma**
- **Pituitary**
- **No tumor**

We chose this dataset for a few reasons:

### ❗A rédiger❗

## 2. Loading the data

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys

sys.path.append("..")

from src.supervised.loader import load_dataset

import numpy as np
import matplotlib.pyplot as plt

X_train, y_train, X_test, y_test = load_dataset("../data/raw/supervised")

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print(f"Train: {len(X_train)} images, {len(y_train)} labels")
print(f"Test: {len(X_test)} images, {len(y_test)} labels")

### 3.1 Sample images per class

In [ ]:
classes = np.unique(y_train)

fig, axes = plt.subplots(4, len(classes), figsize=(8, 10))

for i, classe in enumerate(classes):
    indices = np.where(y_train == classe)[0][:4]
    for j, idx in enumerate(indices):
        axes[i][j].imshow(X_train[idx])
        axes[i][j].set_title(classe)
        axes[i][j].axis("off")

plt.savefig("../figures/supervised/class_samples_overview.png", bbox_inches="tight")
plt.show()

In [ ]:
print(X_train[0].shape)
print(X_train[4200].shape)

Some images appear in grayscale, while others have a purple or blue tint. This isn’t a real color difference, but simply a result of how Matplotlib displays the images. Some are stored as 2D arrays, while others are stored as 3D RGB arrays. During preprocessing, we’ll convert them all to 2D grayscale images. 
As we can see in the cell above, one image has a 3D shape of (221, 228, 3), meaning it contains 3 RGB channels, while the other has a 2D shape of (512, 512), with no additional color channel.

### 3.2 Class distribution

In [ ]:
classes_train, counts_train = np.unique(y_train, return_counts=True)
classes_test, counts_test = np.unique(y_test, return_counts=True)

split_counts = {
    "Train": counts_train,
    "Test": counts_test,
}

width = 0.7

fig, ax = plt.subplots()
bottom = np.zeros(len(classes_train))

for split_name, counts in split_counts.items():
    p = ax.bar(classes_train, counts, width, label=split_name, bottom=bottom)
    bottom += counts

    ax.bar_label(p, label_type="center")

ax.set_title("Class distribution (Train + Test)")
ax.legend()
plt.savefig("../figures/supervised/class_distribution.png", bbox_inches="tight")
plt.show()

The classes are well balanced: each one has around 1400 images in train and 400 in test. This is good news, since we won't need to worry about class imbalance techniques later on when training the model.

### 3.3 Image size check

In [ ]:
images_shapes = [
    image.shape[:2] for image in X_train + X_test
]  # Get only the height and width, ignore channels

unique_shapes, shape_counts = np.unique(images_shapes, axis=0, return_counts=True)

print(f"Number of unique shapes: {len(unique_shapes)}")

heights = [s[0] for s in images_shapes]
widths = [s[1] for s in images_shapes]

print(f"Height range: {min(heights)}px - {max(heights)}px")
print(f"Width range: {min(widths)}px - {max(widths)}px")

sorted_indices = np.argsort(shape_counts)[::-1][:10]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([str(unique_shapes[i]) for i in sorted_indices], shape_counts[sorted_indices])

ax.set_title("Top 10 most common image shapes")
plt.xticks(rotation=45, ha="right")
plt.savefig("../figures/supervised/top10_image_shapes.png", bbox_inches="tight")
plt.show()

print(
    f"{shape_counts[sorted_indices][0]} images have shape {unique_shapes[sorted_indices][0]}"
)

The dataset has 447 different combinations, with heights ranging from 167 to 1446 pixels and widths from 150 to 1375. That's a lot of variation, so resizing all images to a fixed size will definitely be needed before training.

That said, one shape clearly dominates: 5014 images are already (512, 512), out of the 7200 total. So most of the dataset is already consistent, and only a smaller portion needs real resizing.

### 3.4 Grayscale & color check

In [ ]:
combined = X_train + X_test

n_channels = [len(img.shape) for img in combined]
print(
    f"Number of images with 2 channels (grayscale): {n_channels.count(2)}/{len(n_channels)}"
)
print(
    f"Number of images with 3 channels (RGB): {n_channels.count(3)}/{len(n_channels)}"
)

mismatch_indices = []
mean_diffs = []
max_pixel_diffs = []

for i, img in enumerate(combined):
    if len(img.shape) == 3:
        r, g, b = (
            img[:, :, 0].astype(int),
            img[:, :, 1].astype(int),
            img[:, :, 2].astype(int),
        )
        if not (np.array_equal(r, g) and np.array_equal(g, b)):
            mismatch_indices.append(i)
            mean_diffs.append(
                np.mean(
                    [
                        np.abs(r.astype(int) - g.astype(int)).mean(),
                        np.abs(g.astype(int) - b.astype(int)).mean(),
                        np.abs(r.astype(int) - b.astype(int)).mean(),
                    ]
                )
            )
            max_pixel_diffs.append(
                max(np.abs(r - g).max(), np.abs(g - b).max(), np.abs(r - b).max())
            )

print(len(mismatch_indices), "images have mismatched RGB channels")
print(
    f"Average mean channel difference across mismatched images: {np.mean(mean_diffs):.2f}"
)
print(f"Max of these mean differences: {np.max(mean_diffs):.2f}")
print(f"Max single pixel difference: {np.max(max_pixel_diffs)}")

In [ ]:
# Parse the first mismatched image
id = mismatch_indices[0]
img = combined[id]

print(f"Max difference R-G: {np.abs(r.astype(int) - g.astype(int)).max()}")
print(f"Max difference G-B: {np.abs(g.astype(int) - b.astype(int)).max()}")
print(f"Max difference R-B: {np.abs(r.astype(int) - b.astype(int)).max()}")

print(f"Mean difference R-G: {np.abs(r.astype(int) - g.astype(int)).mean():.2f}")
print(f"Mean difference G-B: {np.abs(g.astype(int) - b.astype(int)).mean():.2f}")
print(f"Mean difference R-B: {np.abs(r.astype(int) - b.astype(int)).mean():.2f}")

plt.imshow(img)
plt.title(f"Index {id}")
plt.axis("off")
plt.show()

A "mismatched" image is a 3D (RGB) image where the R, G and B channels aren't perfectly identical, so it's not purely grayscale under the hood. Only 126 images out of 7200 are in this case, which is already a very small portion of the dataset.

Looking at a typical mismatched image, the difference between channels is tiny (0.96 on average) and completely invisible to the naked eye, so it's just normal JPEG compression, nothing worth worrying about.

That said, one single pixel across the whole dataset reaches a difference of 214 out of 255, way too high to be a compression artifact so let's look at the image behind that extreme value.

In [ ]:
outlier_idx_in_list = np.argmax(max_pixel_diffs)
outlier_original_idx = mismatch_indices[outlier_idx_in_list]

print(f"Outlier image index: {outlier_original_idx}")
print(f"Max single-pixel difference: {max_pixel_diffs[outlier_idx_in_list]}/255")

plt.imshow(combined[outlier_original_idx])
plt.title(f"Outlier - Index {outlier_original_idx}")
plt.axis("off")
plt.show()

This image has a colored watermark banner at the bottom (a website URL), which explains the extreme pixel difference. The MRI scan itself is still grayscale, the color only comes from that added banner.

So overall, channel differences in this dataset are either near-zero (JPEG compression noise) or caused by watermarks on a handful of images. Neither carries any real medical information, so converting all 3D images to grayscale by averaging channels during pre-processing is a safe and justified choice.

### 3.5 Train/test duplicates

In [ ]:
test_hashes = set(hash(img.tobytes()) for img in X_test)
overlap = sum(1 for img in X_train if hash(img.tobytes()) in test_hashes)

print(f"Train images also present in test: {overlap}")

### EDA summary

To sum up what we found:
- **Classes are balanced** (1400 train / 400 test each), so no need for rebalancing techniques.
- **Image sizes vary a lot** (447 different combinations, from 150 to 1446 pixels), so resizing to a fixed size is mandatory before training.
- **Images are stored inconsistently**: 3067 as 2D grayscale, 4133 as 3D RGB, even though they're all effectively grayscale. We'll convert everything to 2D.
- **A few images carry a colored watermark**, but they're too rare to justify extra handling.
- **Some test images are exact duplicates of training images**, so we'll drop them from the training set to keep the evaluation honest.


These findings define what we need to do in the pre-processing step below.

## 4. Pre-processing

### 4.1 Grayscale conversion

In [ ]:
from src.supervised.preprocessing import to_grayscale

X_train_gray = [to_grayscale(img) for img in X_train]
X_test_gray = [to_grayscale(img) for img in X_test]

In [ ]:
n_channels = [len(img.shape) for img in X_train_gray + X_test_gray]

print(f"2D images: {n_channels.count(2)}")
print(f"3D images: {n_channels.count(3)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(combined[0])
axes[0].set_title(f"Before - shape {combined[0].shape}")
axes[0].axis("off")

axes[1].imshow(to_grayscale(X_train_gray[0]), cmap="gray")
axes[1].set_title(f"After - shape {to_grayscale(X_train_gray[0]).shape}")
axes[1].axis("off")

plt.show()

### 4.2 Removing leaked images

In [ ]:
test_hashes = set(hash(img.tobytes()) for img in X_test_gray)
keep_mask = np.array([hash(img.tobytes()) not in test_hashes for img in X_train_gray])

X_train_gray = [img for img, keep in zip(X_train_gray, keep_mask) if keep]
y_train = y_train[keep_mask]

print(f"Removed {(~keep_mask).sum()} leaked images")
print(f"Training set: {len(X_train_gray)} images, {len(y_train)} labels")

### 4.3 Testing different sizes

In [ ]:
from src.supervised.preprocessing import resize_image

sizes = [(16, 16), (32, 32), (64, 64), (128, 128)]
sample = X_train_gray[4250]

fig, axes = plt.subplots(1, len(sizes) + 1, figsize=(16, 4))

axes[0].imshow(sample, cmap="gray")
axes[0].set_title(f"Original\n{sample.shape}")
axes[0].axis("off")

for i, size in enumerate(sizes):
    resized = resize_image(sample, size)
    axes[i + 1].imshow(resized, cmap="gray")
    axes[i + 1].set_title(f"{size[0]}x{size[1]}")
    axes[i + 1].axis("off")

plt.tight_layout()
plt.show()

Visually, details start to disappear quickly: at 128x128 the scan still looks clean, at 32x32 most of the texture is gone, and at 16x16 we can barely tell what we're looking at.

But what looks readable to us isn't necessarily what a model relies on. Instead of picking a size based on how it looks, let's test a few sizes with an actual model and compare the results.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import time

for size in sizes:
    X_train_proc = [resize_image(img, size) for img in X_train_gray]
    X_test_proc = [resize_image(img, size) for img in X_test_gray]

    X_train_flat = np.array([img.flatten() for img in X_train_proc])
    X_test_flat = np.array([img.flatten() for img in X_test_proc])

    model = RandomForestClassifier(n_estimators=50, random_state=42)

    start = time.time()
    model.fit(X_train_flat, y_train)
    elapsed = time.time() - start

    accuracy = model.score(X_test_flat, y_test)
    print(f"{size}: accuracy = {accuracy:.4f} ({elapsed:.1f}s)")

The accuracy barely moves across sizes, all around 81%. Even 16x16, where we could barely make out anything by eye, performs just as well as 128x128 while being much faster.

That's suspicious. If the model really needed to see the tumor, shrinking the image down to 16x16 should hurt performance a lot. The fact that it doesn't suggests the model might be relying on something else entirely, something that survives heavy downscaling. Let's look at which parts of the image the model actually uses.

In [ ]:
size = (64, 64)

X_train_proc = [resize_image(img, size) for img in X_train_gray]
X_train_flat = np.array([img.flatten() for img in X_train_proc])

model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train_flat, y_train)

importances = model.feature_importances_.reshape(size)

fig, axe = plt.subplots()
axe.imshow(importances, cmap="hot")
axe.set_title("Feature importance map")
axe.axis("off")

plt.tight_layout()
plt.show()

This confirms the suspicion. The most important pixels form vertical lines along the left and right edges of the image, plus a few scattered points at the top. Almost nothing lights up in the center, where the brain (and the tumor) actually is.

In other words, the model isn't learning to spot tumors. It's learning where the skull ends in the frame, which is a property of how the scan was taken (slice type, machine, patient positioning), not of the pathology itself. That also explains why 16x16 works just as well:
framing information survives heavy downscaling, fine texture doesn't.

This is a classic case of shortcut learning, and it's worth investigating what exactly the model is picking up on.

In [ ]:
classes = np.unique(y_train)
n_samples = 6

fig, axes = plt.subplots(len(classes), n_samples, figsize=(15, 10))

for i, classe in enumerate(classes):
    indices = np.where(y_train == classe)[0][:n_samples]
    for j, idx in enumerate(indices):
        axes[i][j].imshow(X_train_gray[idx], cmap="gray")
        axes[i][j].axis("off")
        if j == 0:
            axes[i][j].set_title(classe, fontsize=10, loc="left")

plt.tight_layout()
plt.show()

Looking at samples per class, the slice orientation clearly isn't random: notumor scans are mostly axial, pituitary ones sagittal, glioma and meningioma coronal or sagittal.

So the model can score high just by recognizing the skull shape in the frame, without ever looking at the tumor. That matches the importance map exactly. Let's crop the edges out and see what happens.

### 4.4 Cropping

In [ ]:
from src.supervised.preprocessing import center_crop


for ratio in [1.0, 0.8, 0.7, 0.6, 0.5]:
    X_train_proc = [center_crop(img, ratio) for img in X_train_gray]
    X_test_proc = [center_crop(img, ratio) for img in X_test_gray]

    X_train_proc = [resize_image(img, size) for img in X_train_proc]
    X_test_proc = [resize_image(img, size) for img in X_test_proc]

    X_train_flat = np.array([img.flatten() for img in X_train_proc])
    X_test_flat = np.array([img.flatten() for img in X_test_proc])

    model = RandomForestClassifier(n_estimators=50, random_state=42)
    model.fit(X_train_flat, y_train)
    accuracy = model.score(X_test_flat, y_test)

    print(f"ratio={ratio}: accuracy = {accuracy:.4f}")

Accuracy drops steadily as we crop more: 81% with no crop, down to 72% at ratio 0.5. That's expected, we're taking away the shortcut. The question is whether the model now looks at the right place.

In [ ]:
ratio = 0.6

X_train_proc = [center_crop(img, ratio) for img in X_train_gray]
X_train_proc = [resize_image(img, size) for img in X_train_proc]
X_train_flat = np.array([img.flatten() for img in X_train_proc])

model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train_flat, y_train)

importances = model.feature_importances_.reshape(size)

fig, axe = plt.subplots(figsize=(12, 5))
axe.imshow(importances, cmap="hot")
axe.set_title(f"Feature importance (crop {ratio})")
axe.axis("off")
plt.tight_layout()
plt.savefig("../figures/supervised/feature_importance_cropped.png", bbox_inches="tight")
plt.show()

After cropping, the importance map looks much healthier: the hot spots moved to two symmetric clusters inside the brain tissue, instead of sitting on the skull outline. Some noise remains in the top corners, where black borders can still leak in depending on the original framing.

The accuracy drops from 87.8% to around 80%, which is expected: we removed the shortcut the model was leaning on. That lower score is probably a more honest estimate of what the model actually learns from brain tissue.

In [ ]:
X_train_crop = [center_crop(img, ratio) for img in X_train_gray]
X_test_crop = [center_crop(img, ratio) for img in X_test_gray]

### 4.5 Resizing

In [ ]:
X_train_resized = np.array([resize_image(img, size) for img in X_train_crop])
X_test_resized = np.array([resize_image(img, size) for img in X_test_crop])

print(f"X_train shape: {X_train_resized.shape}")
print(f"X_test shape: {X_test_resized.shape}")

### 4.6 Normalization

In [ ]:
from src.supervised.preprocessing import normalize

X_train_final = normalize(X_train_resized)
X_test_final = normalize(X_test_resized)

print(
    f"Value range after normalization: {X_train_final.min():.2f} - {X_train_final.max():.2f}"
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(X_train_final[0], cmap="gray")
axes[0].set_title("Normalized image")
axes[0].axis("off")

axes[1].hist(X_train_final[0].ravel(), bins=50)
axes[1].set_title("Pixel value distribution")
axes[1].set_xlabel("Value")

plt.tight_layout()
plt.show()

Normalizing pixel values to 0-1 doesn't change anything for models like Random Forest, which splits on thresholds regardless of scale. We apply it anyway because we plan to compare against SVM, which is distance-based and does need features on a comparable scale to work properly.

## 5. Modeling

### 5.1 Random Forest

<img src="../figures/assets/random_forest.png" width="800">

**Why Random Forest?**

- **Works out of the box on raw pixels.** No feature scaling needed, no assumption about how
  the data is distributed.
- **Hard to break.** Default parameters already give a reasonable result, so we get a
  reference score fast instead of tuning before knowing what we aim at.
- **Fast enough to iterate.** Training on 5485 images takes seconds, which is what made the
  size and crop experiments possible earlier.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

X_train_flat = np.array([img.flatten() for img in X_train_final])
X_test_flat = np.array([img.flatten() for img in X_test_final])

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

rf.fit(X_train_flat, y_train)

y_pred_rf = rf.predict(X_test_flat)

print(classification_report(y_test, y_pred_rf))

76% overall, but the per-class breakdown matters more:

- **pituitary** is the easiest class (F1 0.85), followed by **notumor** (0.82).
- **meningioma** is the weakest (F1 0.62): the model misses 43% of them.
- **notumor** has high recall (0.95) but low precision (0.72), meaning the model labels images as healthy easily, and gets it wrong about a quarter of the time.

In [ ]:
for depth in [None, 20, 30]:
    for max_feat in ["sqrt", "log2", 0.1]:
        rf_test = RandomForestClassifier(
            n_estimators=100,
            max_depth=depth,
            max_features=max_feat,
            random_state=42,
            n_jobs=-1,
        )
        rf_test.fit(X_train_flat, y_train)
        print(
            f"depth={depth}, max_features={max_feat}: {rf_test.score(X_test_flat, y_test):.4f}"
        )

Tuning doesn't help: every combination lands between 74% and 76%, and the defaults already give the best result. The Random Forest has hit its ceiling on this representation.

### 5.2 SVM

<img src="../figures/assets/svm.png" width="400">

**Why SVM?**

- **Different family of algorithm.** Where Random Forest splits on individual pixel thresholds, an SVM looks for a decision boundary in a transformed space, using all features at once. If both reach the same ceiling, that tells us something about the data rather than about the algorithm.
- **Works well in high dimensions.** SVMs are known to hold up when the number of features is large relative to the number of samples, which is our case (4096 features, 5485 images).
- **This is where normalization pays off.** SVMs are distance-based, so unlike Random Forest they genuinely need features on a comparable scale.

In [ ]:
from sklearn.svm import SVC

svm = SVC(kernel="rbf", random_state=42)
svm.fit(X_train_flat, y_train)

y_pred_svm = svm.predict(X_test_flat)
print(classification_report(y_test, y_pred_svm))

With default parameters, the SVM lands at exactly the same 76% as the Random Forest, with a near-identical error profile: pituitary and notumor handled well, meningioma and glioma mixed up.

Two very different algorithms reaching the same score with the same weaknesses is a strong hint that the limit isn't the model, it's the information available in the features.

In [ ]:
for C in [1, 10, 100]:
    for gamma in ["scale", 0.01, 0.001]:
        svm_test = SVC(kernel="rbf", C=C, gamma=gamma, random_state=42)
        svm_test.fit(X_train_flat, y_train)
        print(f"C={C}, gamma={gamma}: {svm_test.score(X_test_flat, y_test):.4f}")

Unlike the Random Forest, the SVM does respond to tuning: accuracy goes from 76% to 81%.

- **C is the main lever.** At C=1 the model is too constrained; raising it to 10 gains 5 points, and C=100 adds almost nothing on top (81.25% vs 81.44%).
- **gamma barely matters**, except 0.001 which degrades every result: the influence of each point becomes too diffuse and the boundaries too smooth.

We keep **C=10, gamma="scale"**: the 0.2 point difference with C=100 isn't meaningful, and a less aggressive model is less likely to have overfitted.

In [ ]:
svm_best = SVC(kernel="rbf", C=10, gamma="scale", random_state=42)
svm_best.fit(X_train_flat, y_train)

y_pred_svm = svm_best.predict(X_test_flat)
print(classification_report(y_test, y_pred_svm))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_rf, cmap="Blues", xticks_rotation=45, ax=axes[0], colorbar=False
)
axes[0].set_title("Random Forest")

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_svm, cmap="Blues", xticks_rotation=45, ax=axes[1], colorbar=False
)
axes[1].set_title("SVM")

plt.tight_layout()
plt.savefig("../figures/supervised/confusion_comparison.png", bbox_inches="tight")
plt.show()

print(f"Random Forest: {rf.score(X_test_flat, y_test):.4f}")
print(f"SVM: {svm_best.score(X_test_flat, y_test):.4f}")

The tuned SVM is our best model at 81%. Beyond the score, the confusion matrices show both models share the same blind spot: glioma and meningioma get confused with each other, and real tumors end up predicted as notumor.

This lines up with what we found in pre-processing. notumor scans are mostly axial and pituitary ones mostly sagittal, so those classes carry an orientation cue that survives even after cropping. Glioma and meningioma share similar orientations, so telling them apart requires actually reading the tissue, which is where both models struggle.